# Método de Trabalho

Bibliotecas utilizadas no desenvolvimento do código

In [ ]:
import pandas as pd

from langdetect import detect
from collections import Counter

from urlextract import URLExtract
import re

from nltk.corpus import words
import nltk
nltk.download('words')

import emoji
from emot.emo_unicode import EMOTICONS_EMO

import unicodedata

import contractions

from spellchecker import SpellChecker

from sklearn.model_selection import train_test_split

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

from sentence_transformers import SentenceTransformer

import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
import ast

## Coleta dos Dados

In [ ]:
df = pd.read_csv('datasets/cyberbullying_tweets.csv')

df

In [ ]:
pd.set_option('display.max_colwidth', None)

print(df.loc[135, 'tweet_text'])
print(df.loc[29538, 'tweet_text'])
print(df.loc[13895, 'tweet_text'])
print(df.loc[160, 'tweet_text'])
print(df.loc[200, 'tweet_text'])
print(df.loc[64, 'tweet_text'])
print(df.loc[224, 'tweet_text'])
print(df.loc[44, 'tweet_text'])

## Pré-processamento dos Dados

### Limpeza

#### URLs

In [ ]:
# Remover URLs

extractor = URLExtract()

def remover_urls(texto):
    texto = str(texto)

    urls = extractor.find_urls(texto)

    for url in urls:
        texto = texto.replace(url, '')

    return texto

In [ ]:
# Verificar ocorrências de URLs

extractor = URLExtract()

def inspecionar_urls(linha):
    texto = str(linha['tweet_text'])

    urls = extractor.find_urls(texto)

    if urls:
        print(f"Índice: {linha.name} | Texto: {texto}")

df.apply(inspecionar_urls, axis=1)

#### E-mails

In [ ]:
# Remover e-mails

pattern_emails = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

def remover_emails(texto):
    texto = str(texto)
    texto_limpo = pattern_emails.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de e-mails

pattern_emails = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

def inspecionar_emails(linha):
    texto = str(linha['tweet_text'])
    emails = pattern_emails.findall(texto)

    if emails:
        print(f"Índice: {linha.name} | Texto: {texto}")

df.apply(inspecionar_emails, axis=1)

#### Menção a user

In [ ]:
# Remover menções a users

pattern_users = re.compile(r'(?:^| )(@[A-Za-z0-9_@]+)')

def remover_users(texto):
    texto = str(texto)
    texto_limpo = pattern_users.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de menções a users

pattern_users = re.compile(r'(?:^| )(@[A-Za-z0-9_]+)')

def inspecionar_users(linha):
    texto = str(linha['tweet_text'])
    users = pattern_users.findall(texto)

    if users:
        print(f"Índice: {linha.name} | Texto: {texto}")

df.apply(inspecionar_users, axis=1)

#### Espaços entre caracteres únicos consecutivos

In [ ]:
# Remover espaços entre caracteres únicos consecutivos

pattern_espacos = re.compile(r'(?:\b\w\s){3,}\w\b')

def remover_espacos(texto):
    texto = str(texto)
    texto_limpo = pattern_espacos.sub(lambda x: x.group().replace(' ', ''), texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de espaços entre caracteres únicos consecutivos

pattern_espacos = re.compile(r'(?:\b\w\s){3,}\w\b')

def inspecionar_espacos(linha):
    texto = str(linha['tweet_text'])
    espaco = pattern_espacos.findall(texto)

    if espaco:
        print(f"Índice: {linha.name} | Texto: {texto}")

df.apply(inspecionar_espacos, axis=1)

#### Múltiplos espaços

In [ ]:
# Remover múltiplos espaços

def remover_multiplos_espacos(texto):
    return ' '.join(texto.split())

In [ ]:
# Verificar ocorrências de múltiplos espaços

pattern_multiplos_espacos = re.compile(r'\s{2,}')

def inspecionar_multiplos_espacos(linha):
    texto = str(linha['tweet_text'])
    espaco = pattern_multiplos_espacos.findall(texto)

    if espaco:
        print(f"Índice: {linha.name} | Texto: {texto}")

df.apply(inspecionar_multiplos_espacos, axis=1)

#### Filtrar idiomas

In [ ]:
# Inspecionar idiomas presentes no df

def detectar_idioma(texto):
    try:
        return detect(str(texto))
    except:
        return 'desconhecido'

idiomas = df['tweet_text'].apply(detectar_idioma)
contagem = Counter(idiomas)

for idioma, quantidade in contagem.most_common():
    print(f'{idioma}: {quantidade} tweets ({quantidade/len(df)*100:.1f}%)')

In [ ]:
# Manter apenas dados da língua inglesa

def detectar_ingles(texto):
    try:
        return detect(str(texto)) == 'en'
    except:
        return False

#### Encapsulamento do pré-processamento 01

In [ ]:
def preprocessar_01(texto):
    texto = remover_urls(texto)
    texto = remover_emails(texto)
    texto = remover_users(texto)
    texto = remover_espacos(texto)
    texto = remover_multiplos_espacos(texto)

    return texto

df['tweet_text'] = df['tweet_text'].apply(preprocessar_01)
df = df[df['tweet_text'].apply(detectar_ingles)]

### Amostragem

In [ ]:
# Amostragem estratificada - 20% para classes problemáticas, 10% para as demais

partes = []

for classe, grupo in df.groupby('cyberbullying_type'):
    frac = 0.20 if classe in ['not_cyberbullying', 'other_cyberbullying'] else 0.10
    partes.append(grupo.sample(frac=frac, random_state=42))

df_rotulado = pd.concat(partes)
df_nao_rotulado = df.drop(df_rotulado.index)

df_nao_rotulado_backup = df_nao_rotulado.copy()
df_nao_rotulado['cyberbullying_type'] = None

print(f'Total: {len(df_rotulado)}')
print(df_rotulado['cyberbullying_type'].value_counts())

print(f'\nTotal: {len(df_nao_rotulado)}')
print(df_nao_rotulado['cyberbullying_type'].value_counts())

### Normalização

#### Normalização de caracteres alongados

In [ ]:
# Remover caracteres alongados

palavras_validas = set(words.words())
pattern_caracteres_alongados = re.compile(r'(\w)\1{1,}')

def remover_caracteres_alongados(texto):
    texto = str(texto)
    tokens = texto.split()
    resultado = []

    for token in tokens:
        if token.lower() not in palavras_validas:
            token = pattern_caracteres_alongados.sub(r'\1', token)
        resultado.append(token)

    return ' '.join(resultado)

In [ ]:
# Verificar ocorrências de caracteres alongados

pattern_caracteres_alongados = re.compile(r'(\w)\1{2,}')

def inspecionar_caracteres_alongados(linha):
    texto = str(linha['tweet_text'])
    espaco = pattern_caracteres_alongados.findall(texto)

    if espaco:
        print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_caracteres_alongados, axis=1)
df_nao_rotulado.apply(inspecionar_caracteres_alongados, axis=1)

#### Transformação de emojis e emoticons em texto

In [ ]:
# Converter emoticons em texto

pattern_emoticons = re.compile('|'.join(re.escape(k) for k in sorted(EMOTICONS_EMO, key=len, reverse=True)))

def converter_emoticons(texto):
    return pattern_emoticons.sub(lambda m: EMOTICONS_EMO[m.group()], str(texto))

In [ ]:
# Verificar ocorrências de emoticons

def inspecionar_emoticons(linha):
    texto = str(linha['tweet_text'])
    encontrados = [emoticon for emoticon in EMOTICONS_EMO if emoticon in texto]
    
    if encontrados:
        print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_emoticons, axis=1)
df_nao_rotulado.apply(inspecionar_emoticons, axis=1)

In [ ]:
# Converter emojis em texto

def converter_emojis(texto):
    return emoji.demojize(str(texto))

In [ ]:
# Verificar ocorrências de emojis

def inspecionar_emojis(linha):
    texto = str(linha['tweet_text'])

    if emoji.emoji_count(texto) > 0:
        print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_emojis, axis=1)
df_nao_rotulado.apply(inspecionar_emojis, axis=1)

#### Caracteres acentuados normalizados para o alfabeto inglês

In [ ]:
# Remover acentos

def remover_acentos(texto):
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    
    return texto

#### Caracteres para Minúsculo

In [ ]:
# to_lowercase

def to_lowercase(texto):
    return str(texto).lower()

#### Conversão de gírias para palavras normais

In [ ]:
# Criar dicionário de gírias

def criar_girias(arquivo):
    dicionario_girias = {}

    df_girias = pd.read_csv(arquivo)
    dicionario_girias.update(dict(zip(df_girias['giria'].str.lower(), df_girias['significado'].str.lower())))

    return dicionario_girias

In [ ]:
# Converter gírias para palavras normais

def converter_girias(texto, dicionario_girias):
    texto = str(texto)
    tokens = texto.split()
    resultado = []

    for token in tokens:
        if token.lower() not in palavras_validas:
            token = dicionario_girias.get(token.lower(), token)
            
        resultado.append(token)
    
    return ' '.join(resultado)

In [ ]:
# Verificar ocorrências de gírias

dicionario_girias = criar_girias('datasets/girias.csv')

def inspecionar_girias(linha):
    texto = str(linha['tweet_text'])
    tokens = texto.split()

    for token in tokens:
         if token.lower() not in palavras_validas:
               if dicionario_girias.get(token.lower()):
                    print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_girias, axis=1)
df_nao_rotulado.apply(inspecionar_girias, axis=1)

#### Expansão de contrações

In [ ]:
# Expandir contrações

def expandir_contracoes(texto):
    texto = str(texto)
    texto_expandido = contractions.fix(texto)
    
    return texto_expandido

In [ ]:
# Verificar ocorrências de contrações

lista_contracoes_reais = [
    chave for chave, valor in contractions.contractions_dict.items() 
    if chave.lower() != valor.lower()
]

regex_string = r'\b(' + '|'.join([re.escape(chave) for chave in lista_contracoes_reais]) + r')\b'
pattern_contracoes = re.compile(regex_string, flags=re.IGNORECASE)

def inspecionar_contracoes(linha):
    texto = str(linha['tweet_text'])
    contracoes_encontradas = pattern_contracoes.findall(texto)
    
    if contracoes_encontradas:
        print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_contracoes, axis=1)
df_nao_rotulado.apply(inspecionar_contracoes, axis=1)

#### Encapsulamento do pré-processamento 02

In [ ]:
arquivo = 'datasets/girias.csv'
palavras_validas = set(words.words())
dicionario_girias = criar_girias('datasets/girias.csv')

def preprocessar_02(texto, dicionario_girias):
    texto = remover_caracteres_alongados(texto)
    texto = converter_emoticons(texto)
    texto = converter_emojis(texto)
    texto = remover_acentos(texto)
    texto = to_lowercase(texto)
    texto = converter_girias(texto, dicionario_girias)
    texto = expandir_contracoes(texto)

    return texto

df_rotulado['tweet_text'] = df_rotulado['tweet_text'].apply(lambda x: preprocessar_02(x, dicionario_girias))
df_nao_rotulado['tweet_text'] = df_nao_rotulado['tweet_text'].apply(lambda x: preprocessar_02(x, dicionario_girias))

### Remoção

#### Remoção de números

In [ ]:
# Remover números

pattern_numeros = re.compile(r'\d+')

def remover_numeros(texto):
    texto = str(texto)
    texto_limpo = pattern_numeros.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de números

pattern_numeros = re.compile(r'\d+')

def inspecionar_numeros(linha):
    texto = str(linha['tweet_text'])
    numeros = pattern_numeros.findall(texto)

    if numeros:
        print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_numeros, axis=1)
df_nao_rotulado.apply(inspecionar_numeros, axis=1)

#### Remoção de pontuações

In [ ]:
# Remover pontuações

pattern_pontuacoes = re.compile(r'[^\w\s]')

def remover_pontuacoes(texto):
    texto = str(texto)
    texto_limpo = pattern_pontuacoes.sub('', texto)

    return texto_limpo

In [ ]:
# Verificar ocorrências de pontuações

pattern_pontuacoes = re.compile(r'[^\w\s]')

def inspecionar_pontuacoes(linha):
    texto = str(linha['tweet_text'])
    pontuacoes = pattern_pontuacoes.findall(texto)

    if pontuacoes:
        print(f"Índice: {linha.name} | Texto: {texto}")

df_rotulado.apply(inspecionar_pontuacoes, axis=1)
df_nao_rotulado.apply(inspecionar_pontuacoes, axis=1)

#### Remoção de múltiplos espaços

In [ ]:
# Remover múltiplos espaços

def remover_multiplos_espacos(texto):
    return ' '.join(texto.split())

In [ ]:
# Verificar ocorrências de múltiplos espaços

pattern_multiplos_espacos = re.compile(r'\s{2,}')

def inspecionar_multiplos_espacos(linha):
    texto = str(linha['tweet_text'])
    espaco = pattern_multiplos_espacos.findall(texto)

    if espaco:
        print(f"Índice: {linha.name} | Texto: {texto}")

df.apply(inspecionar_multiplos_espacos, axis=1)

#### Encapsulamento do pré-processamento 03

In [ ]:
def preprocessar_03(texto):
    texto = remover_numeros(texto)
    texto = remover_pontuacoes(texto)
    texto = remover_multiplos_espacos(texto)

    return texto

df_rotulado['tweet_text'] = df_rotulado['tweet_text'].apply(preprocessar_03)
df_nao_rotulado['tweet_text'] = df_nao_rotulado['tweet_text'].apply(preprocessar_03)

### Lematização

In [ ]:
# Salva uma cópia dos tweets originais para comparar com o df após a lematização

df_rotulado['tweet_original'] = df_rotulado['tweet_text']
df_nao_rotulado['tweet_original'] = df_nao_rotulado['tweet_text']

In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lematizar(texto):
    texto = str(texto)
    tokens = nltk.word_tokenize(texto)
    tags = nltk.pos_tag(tokens)
    resultado = []
    
    for token, tag in tags:
        if tag in ('PRP', 'PRP$'):
            continue
        pos = get_wordnet_pos(tag)
        lema = lemmatizer.lemmatize(token, pos)

        if lema == 'be':
            continue
        
        resultado.append(lema)
    
    return ' '.join(resultado)

In [ ]:
df_rotulado['tweet_text'] = df_rotulado['tweet_text'].apply(lematizar)
df_nao_rotulado['tweet_text'] = df_nao_rotulado['tweet_text'].apply(lematizar)

## Representação Vetorial dos Dados

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

tweets_rotulados = df_rotulado['tweet_text'].tolist()
embeddings_rotulados = model.encode(tweets_rotulados)

tweets_nao_rotulados = df_nao_rotulado['tweet_text'].tolist()
embeddings_nao_rotulados = model.encode(tweets_nao_rotulados)

print(f'Embeddings rotulados: {embeddings_rotulados.shape}')
print(f'Embeddings não rotulados: {embeddings_nao_rotulados.shape}')

## Armazenamento dos Dados

In [ ]:
# Executar script criacao_db_tabelas.sql no PostgreSQL antes de prosseguir

In [ ]:
try:
    conn = psycopg2.connect(
        host = 'localhost',
        database = 'CyberbullyingDetection',
        user = 'postgres',
        password = 'april2104'
    )

    cursor = conn.cursor()
    register_vector(conn)

    cursor.execute("TRUNCATE TABLE dados_rotulados RESTART IDENTITY;")
    cursor.execute("TRUNCATE TABLE dados_nao_rotulados RESTART IDENTITY;")

    rotulos = df_rotulado['cyberbullying_type'].tolist()
    registros_rotulados = list(zip(tweets_rotulados, embeddings_rotulados, rotulos))
    registros_nao_rotulados = list(zip(tweets_nao_rotulados, embeddings_nao_rotulados, df_nao_rotulado.index))

    execute_values(cursor, """
                INSERT INTO dados_rotulados (tweet, embedding, rotulo)
                VALUES %s
    """, registros_rotulados)

    execute_values(cursor, """
                INSERT INTO dados_nao_rotulados (tweet, embedding, indice_original)
                VALUES %s
    """, registros_nao_rotulados)

    conn.commit()
    print(f"{len(registros_rotulados) + len(registros_nao_rotulados)} registros inseridos com sucesso")

except Exception as e:
    conn.rollback()
    print(f"Erro: {e}")

finally:
    cursor.close()
    conn.close()

## Classificação e Avaliação

In [ ]:
# Algoritmo de classificação kNN no PostgreSQL

k = 7
valores_k = [3, 5, 7, 9, 11, 15, 21, 31, 51]
resultados = []

for k in valores_k:
    print(f"Executando kNN com k = {k}")

    # Classificação
    try:
        conn = psycopg2.connect(
            host = 'localhost',
            database = 'CyberbullyingDetection',
            user = 'postgres',
            password = 'april2104'
        )

        cursor = conn.cursor()
        register_vector(conn)

        cursor.execute("SELECT classificar_knn(%s)", (k,))
        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Erro: {e}")

    finally:
        cursor.close()
        conn.close()


    # Avaliação
    try:
        conn = psycopg2.connect(
            host = 'localhost',
            database = 'CyberbullyingDetection',
            user = 'postgres',
            password = 'april2104'
        )

        cursor = conn.cursor()

        cursor.execute("SELECT id, tweet, rotulo, indice_original FROM dados_nao_rotulados")
        resultados = cursor.fetchall()

        df_resultado = pd.DataFrame(resultados, columns=['id', 'tweet', 'rotulo_knn', 'indice_original'])

    except Exception as e:
        print(f"Erro: {e}")

    finally:
        cursor.close()
        conn.close()

    df_comparacao = df_resultado.merge(
        df_nao_rotulado_backup[['cyberbullying_type']],
        left_on = 'indice_original',
        right_index = True,
        how = 'left'
    )

    df_real = df_comparacao['cyberbullying_type']
    df_pred = df_comparacao['rotulo_knn']

    print(classification_report(df_real, df_pred))

## Matriz de Confusão

In [ ]:
try:
    conn = psycopg2.connect(
        host='localhost',
        database='CyberbullyingDetection',
        user='postgres',
        password='april2104'
    )
    cursor = conn.cursor()
    cursor.execute("SELECT id, tweet, rotulo, indice_original FROM dados_nao_rotulados")
    rows = cursor.fetchall()
    df_resultado = pd.DataFrame(rows, columns=['id', 'tweet', 'rotulo_knn', 'indice_original'])

except Exception as e:
    print(f"Erro: {e}")
finally:
    cursor.close()
    conn.close()

df_comparacao = df_resultado.merge(
    df_nao_rotulado_backup[['cyberbullying_type']],
    left_on='indice_original',
    right_index=True,
    how='left'
)

y_real = df_comparacao['cyberbullying_type']
y_pred = df_comparacao['rotulo_knn']
classes = sorted(y_real.unique())

cm = confusion_matrix(y_real, y_pred, labels=classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Matriz de Confusão (valores absolutos)')
axes[0].set_ylabel('Classe Real')
axes[0].set_xlabel('Classe Predita')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Matriz de Confusão (normalizada por linha)')
axes[1].set_ylabel('Classe Real')
axes[1].set_xlabel('Classe Predita')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP e t-SNE

In [ ]:
try:
    conn = psycopg2.connect(
        host='localhost',
        database='CyberbullyingDetection',
        user='postgres',
        password='april2104'
    )
    cursor = conn.cursor()
    register_vector(conn)

    cursor.execute("SELECT embedding, rotulo FROM dados_rotulados")
    rows = cursor.fetchall()
    print(f"{len(rows)} embeddings carregados.")

except Exception as e:
    print(f"Erro: {e}")
finally:
    cursor.close()
    conn.close()

embeddings = np.array([np.array(row[0]) for row in rows])
rotulos = [row[1] for row in rows]

cores = {
    'age': '#2196F3',
    'ethnicity': '#4CAF50',
    'gender': '#FF9800',
    'not_cyberbullying': '#F44336',
    'other_cyberbullying': '#9C27B0',
    'religion': '#00BCD4'
}

cores_lista = [cores[r] for r in rotulos]
classes = list(cores.keys())

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
emb_umap = reducer.fit_transform(embeddings)

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
emb_tsne = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, emb, titulo in zip(axes, [emb_umap, emb_tsne], ['UMAP', 't-SNE']):
    for classe in classes:
        idx = [i for i, r in enumerate(rotulos) if r == classe]
        ax.scatter(
            emb[idx, 0], emb[idx, 1],
            c=cores[classe], label=classe,
            alpha=0.4, s=5
        )
    ax.set_title(f'Embeddings SentenceTransformer — {titulo}', fontsize=13)
    ax.legend(markerscale=3, fontsize=9)
    ax.set_xlabel('Dimensão 1')
    ax.set_ylabel('Dimensão 2')

plt.tight_layout()
plt.show()